# Imports and Setup

In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

In [ ]:
from skewerjax.skewers.skewers import SkewerBundle
from skewerjax.cosmology.cosmology import Cosmology

from skewerjax.fields.fields import ScalarField, VectorField, Fields
import skewerjax.fields.toy_model_samplers as toy_samplers
import skewerjax.utils.aux as aux

from skewerjax.optical_depth import line_profile
import skewerjax.optical_depth.optical_depth as od

from skewerjax.constants.physical_constants import phys
from skewerjax.optical_depth.voigt import Lorentzian
from skewerjax.optical_depth.voigt import Gaussian

In [ ]:
# Create cosmology object
cosmology = Cosmology(
    H0=phys.H0,
    Omega_r_0=1.0,  # Radiation density (CMB + neutrinos)
    Omega_m_0=0.0,
    Omega_k_0=0.0,
    Omega_lambda_0=0.0,
)

# Define which cosmology to use, this is set as default to 0 since skewer and cosmology objects can deal with multiple cosmologies at once, 
# but other classes such as fields and optical depth are currently only set up for a single cosmology at a time

ii_cosmology = 0

In [ ]:
wavelengths = jnp.linspace(970, 1230, 2**17)  # in Angstroms

line_bundle = sjx.constants.lines.LineBundle(["HI-LyA", "HI-LyB", "SiIII-1206"])
line_bundle_null = sjx.constants.lines.LineBundle(["nullHI-LyA", "nullHI-LyB", "nullSiIII1206"])

In [ ]:
T0_tiny = jnp.finfo(jnp.dtype('float32')).tiny

T0_min = 1e4
T0_max = 1e8

n0_min = 1e-9
n0_max = 1e-3
n0 = 1e-7

v0 = 0.0

L_box_Kpc      = 1e-3 # in Mpc
L_box          = 1    # in Mpc
L_box_five_Mpc = 5    # in Mpc

In [ ]:
line_bundle = sjx.constants.lines.LineBundle(["HI-LyA"])

In [ ]:
fields = Fields({
    "HI": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_min=T0_tiny, T0_max=T0_tiny, L_box=L_box)),            # UNITS: [K]
        "velocity": VectorField(toy_samplers.velocity_sampler(v0_min=v0, v0_max=v0, L_box=L_box)),                  # UNITS: [km/s]
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),                                     # UNITS: [km/s]
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0)),                                   # UNITS: [GHz]
        "number_density": ScalarField(toy_samplers.number_density_sampler(n0_min=1e-9, n0_max=4e-9, L_box=L_box)),  # UNITS: [cm^-3]
    }
})

In [ ]:
N_sk = 2

In [ ]:
_delta = 0.35 # [Angstrom]
wavelengths = jnp.linspace(line_bundle.lambda0_A[0]-6e-2, line_bundle.lambda0_A[0]+_delta, 2**17)  # in Angstroms

N_resolution_ppsk = 2
array_N_ppsk = jnp.linspace(16, 31, N_resolution_ppsk).astype(jnp.int32)

F_over_F0 = []
for ii_N_ppsk, N_ppsk in enumerate(array_N_ppsk):

    _points        = aux.make_plane_points(L_box, 'x', L_box/2, N_sk, N_ppsk)
    skewer_bundle = SkewerBundle(
        spatial_points=_points,
        a0=1.0,                                                      # scale factor at which ICs are defined, default: 1.0
        x0=0.0,                                                      # comoving distance at which ICs are defined, default: 0.0
        ICs_defined_at_index=-1,                                     # indices along the skewer where ICs are defined, (-1 means last point, 0 means first point), default: -1
        ICs_at_emission_or_absorption="absorption",                  # whether the ICs are defined at emission or absorption, default: 'absorption'
        cosmologies=cosmology                                        # when the manual scale factors are not provided, cosmologies must be defined
    )

    optical_depth = od.compute_optical_depth(
        skewer_bundle,
        wavelengths,
        line_bundle,
        fields,
        compute_mode="vmap",
        num_gpus=None
    )
    F_over_F0.append(jnp.exp(-optical_depth))

F_over_F0 = jnp.stack(F_over_F0, axis=0)
print(F_over_F0.shape) # (N_resolution_ppsk, N_lines, N_sk, N_lambda)

In [ ]:
optical_depth_analytic = od.compute_optical_depth_analytical_radiation_domination_Natural(
    wavelengths,
    line_bundle.lambda0_A,
    line_bundle.A21_GHz,
    line_bundle.f_osc,
    skewer_bundle.a0,
    skewer_bundle.x0,
    xem_Mpc=-L_box * jnp.ones(N_sk),
    xf_Mpc=jnp.zeros(N_sk),
    n0_cm3=fields.number_density(skewer_bundle.quadri_vectors[ii_cosmology])[0, :, 0]
)

F_over_F0_analytic = jnp.exp(-optical_depth_analytic)

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(wavelengths)                             # (N_lambda,)
F_over_F0_np = np.asarray(F_over_F0)                             # (resolution N_ppsk, N_lines, N_sk, N_lambda)

colors = np.array([
    plt.cm.Reds(np.linspace(0.4, 0.8, N_resolution_ppsk)),
    plt.cm.Greens(np.linspace(0.4, 0.8, N_resolution_ppsk))
])

# Create figure with two subplots (top: original, bottom: residuals)
fig, (ax_top, ax_res) = plt.subplots(2, 1, figsize=(15, 10), 
                                      height_ratios=[2, 1], 
                                      sharex=True)

# Plot all profiles on the same axes in top panel
handles_colors = []
handles_N_ppsk = []
for i_sp, line in enumerate(line_bundle.lines):
    for i_sk in range(N_sk):
        ax_top.plot(lambdas_np, F_over_F0_analytic[i_sp, i_sk, :], color=colors[i_sk][-1], linewidth=4.0, linestyle='-', alpha=0.7)
        for i_ppsk, N_ppsk in enumerate(array_N_ppsk):
            ax_top.plot(lambdas_np, F_over_F0_np[i_ppsk, i_sp, i_sk, :], color=colors[i_sk, i_ppsk], linewidth=1.5, linestyle='--')

            if i_sp == 0:
                _lbl = r"$N_\mathrm{ppsk} =$" + str(round(N_ppsk, 2))
                handles_N_ppsk.append(Line2D([0], [0], color=colors[i_sk, i_ppsk], lw=4, linestyle="-", label=_lbl))
        
        if i_sp == 0:
            _sampled_numbr_density_cm3 = fields.number_density(skewer_bundle.quadri_vectors[ii_cosmology])[i_sp, i_sk, 0]
            _lbl = rf"$n = {_sampled_numbr_density_cm3:.2g} \, \mathrm{{cm^{{-3}}}}$"
            handles_colors.append(Line2D([0], [0], color=colors[i_sk][-1], lw=4, linestyle="-", label=_lbl))

        # Plot analytic reference line at zero
        if i_sp == 0:
            ax_res.axhline(y=0, color=colors[i_sk][-1], linestyle='-', linewidth=2.0, alpha=0.3)
        
        for i_ppsk, N_ppsk in enumerate(array_N_ppsk):
            residual = F_over_F0_np[i_ppsk, i_sp, i_sk] - F_over_F0_analytic[i_sp, i_sk]
            ax_res.plot(lambdas_np, residual, color=colors[i_sk, i_ppsk], linestyle='--', linewidth=1.5)

    ax_top.axvline(line.lambda0_A, color="k", linestyle="-", linewidth=2., alpha=1.0)

    # Add vertical line at line center
    ax_res.axvline(line.lambda0_A, color="k", linestyle="-", linewidth=2., alpha=0.5)

# Legends for top panel (keeping your original legend code)
leg = ax_top.legend(handles=handles_colors, loc="lower right", fontsize=16, ncols=1)
ax_top.add_artist(leg)

leg = ax_top.legend(handles=handles_N_ppsk, loc="upper center", fontsize=16, ncols=len(array_N_ppsk))
ax_top.add_artist(leg)

handles_linestyles = [
    Line2D([0], [0], color="grey", lw=4, linestyle="--", label="SkewerJAX"),
    Line2D([0], [0], color="grey", lw=4, linestyle="-", label="Analytical")
]
leg = ax_top.legend(handles=handles_linestyles, loc="lower left", fontsize=16, ncols=1)
ax_top.add_artist(leg)

ax_res.fill_between(lambdas_np, 0.1, -0.1, color="grey", alpha=0.2)
ax_res.hlines(0.1, xmin=lambdas_np[0], xmax=lambdas_np[-1], color="k", linewidth=1, linestyle='--', alpha=0.5)
ax_res.hlines(-0.1, xmin=lambdas_np[0], xmax=lambdas_np[-1], color="k", linewidth=1, linestyle='--', alpha=0.5)

# Axes formatting for top panel
ax_top.set_ylabel(r"$F(\lambda) / F_0\, \left[\mathrm{adim}\right]$")

# Axes formatting for residual panel
ax_res.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
ax_res.set_ylabel(r"Num. - Analytic")

ax_top.set_title(r"Radiation domination." + r" $L_\mathrm{box} = $" + str(L_box) + r"$\,\mathrm{Mpc}.$" + r"$.\, a_0 = $" + str(round(skewer_bundle.a0[0], 4)), fontsize=20)

# Adjust layout
fig.tight_layout()
plt.show()

\begin{align}
-\ln f
=
\frac{
2 n_0 \mathcal{C}_j \left\lbrace \Delta x \right\rbrace
}{
\left[ a_0 + \frac{H_0 \Delta x}{2 c} \right]^{2}
}
V \left( \frac{c}{\lambda_j} \left[ 1 - \frac{a_0}{ a_0 + \frac{H_0 \Delta x}{2 c} } \right] ;\; \sigma_j ,\, \gamma_j  \right)
\end{align}

In [ ]:
from skewerjax.optical_depth.voigt import log10_voigt

In [ ]:
def compute_transcendental_resolution_function(
    line_bundle: sjx.constants.lines.LineBundle,
    T0_K: jnp.ndarray,
    n0_cm3: jnp.ndarray,
    Delta_x_Mpc: jnp.ndarray,
    a0: float,
    method: str = "trapezoidal"
):    
    f_osc = line_bundle.f_osc           # N_lines
    lambda0_A = line_bundle.lambda0_A   # N_lines
    A21_GHz = line_bundle.A21_GHz       # N_lines
    mass_amu = line_bundle.mass_amu     # N_lines

    if method == "trapezoidal":
        _term = Delta_x_Mpc
    elif method == "analytical_deltas":
        _term = 1.0
    else:
        raise NotImplementedError
    
    N_lines = f_osc.shape[0]
    N_T0 = T0_K.shape[0]
    N_xx = Delta_x_Mpc.shape[0]

    # --- Broadcast to common shape: (N_lines, N_T0, N_n0, N_xx) ---
    T0_K = T0_K[None, :, None, None]
    n0_cm3 = n0_cm3[None, None, :, None]
    f_osc = f_osc[:, None, None, None]
    lambda0_A = lambda0_A[:, None, None, None]
    A21_GHz = A21_GHz[:, None, None, None]
    mass_amu = mass_amu[:, None, None, None]
    Delta_x_Mpc = Delta_x_Mpc[None, None, None, :]

    v_eff_A_GHz = jnp.sqrt(2.0 * phys.kB * T0_K / mass_amu) * 10.0
    sigma_GHz = v_eff_A_GHz / (2.0 * lambda0_A) # (N_lines, N_T0, 1, 1)
    gamma_GHz = A21_GHz / (4.0 * jnp.pi)        # (N_lines, 1, 1, 1)

    # Tile gamma_GHz to (N_lines * N_T0), then flatten to (N_lines * N_T0)
    gamma_GHz_tiled = jnp.tile(gamma_GHz, (1, N_T0, 1, 1))
    gamma_GHz_flat = gamma_GHz_tiled.reshape(-1)
    # Flatten sigma_GHz to (N_lines * N_T0)
    sigma_GHz_flat = sigma_GHz.reshape(-1)
    # flatten Delta_xx_Mpc
    nu_GHz = (phys.c * 10.0 / lambda0_A) * (1 - a0 / (a0 + 0.5 * phys.H0_over_c * Delta_x_Mpc)) # GHz, (N_lines, 1, 1, N_xx)
    nu_GHz_tiled = jnp.tile(nu_GHz, (1, N_T0, 1, 1))
    nu_flat_GHz = jnp.reshape(nu_GHz_tiled, (N_lines*N_T0, N_xx))
    # compute phi using Voigt profile
    voigt_batched = vmap(lambda x, s, g: 10**log10_voigt(x, s, g), in_axes=(0, 0, 0))
    phi_flat_ns = voigt_batched(nu_flat_GHz, sigma_GHz_flat, gamma_GHz_flat) # (N_lines * N_T0, N_xx)
    # Reshape to (N_lines, N_T0, 1, N_xx)
    phi_ns = phi_flat_ns.reshape(N_lines, N_T0, 1, N_xx)

    C_j_cm2_GHz = phys.cross_section_prefactor * f_osc
    rhs = (2 * n0_cm3 * C_j_cm2_GHz * _term) * (1e2 * phys.Mpc_to_m) # GHz
    rhs *= (a0 + 0.5 * phys.H0_over_c * Delta_x_Mpc)**-2             # GHz
    rhs *= phi_ns                                                    # adim

    return rhs

In [ ]:
line_bundle = sjx.constants.lines.LineBundle(["HI-LyA", "HI-LyB", "SiIII-1206"])

T0_min = 1e1
T0_max = 1e4
N_T0 = 3

n0_min = 1e-9
n0_max = 1e-5
N_n0 = 5

T0 = jnp.geomspace(T0_min, T0_max, N_T0)
n0 = jnp.geomspace(n0_min, n0_max, N_n0)
Delta_x = jnp.geomspace(1e-8, 0.5, 10000)
a0 = 1.0
resolution_fraction = 0.1

transcendental_resolution = compute_transcendental_resolution_function(
    line_bundle, T0, n0, Delta_x, a0
)

transcendental_resolution = np.array(transcendental_resolution)
Delta_x = np.array(Delta_x)

In [ ]:
N_lines = len(line_bundle.lambda0_A)
fixed_n0_idx = 0
fixed_T0_idx = 0

fig, axes = plt.subplots(2, N_lines, figsize=(4*N_lines, 8), constrained_layout=True)

cmap1 = plt.cm.viridis(np.linspace(0.2, 0.9, len(T0)))  # T0
cmap2 = plt.cm.plasma(np.linspace(0.2, 0.9, len(n0)))   # n0

# -------------------------
# Build legend handles once
# -------------------------
handles_T0 = []
for ii_T0, col in enumerate(cmap1):
    _lbl = rf"$T_0 = {T0[ii_T0]:.3g}\,\mathrm{{K}}$"
    handles_T0.append(Line2D([0], [0], color=col, lw=4, linestyle="-", label=_lbl))

handles_n0 = []
for ii_n0, col in enumerate(cmap2):
    _lbl = rf"$n_0 = {n0[ii_n0]:.3g}\,\mathrm{{cm^{{-3}}}}$"
    handles_n0.append(Line2D([0], [0], color=col, lw=4, linestyle="-", label=_lbl))

# -------------------------
# Top row: vary T0 (fix n0)
# -------------------------
for ii_line in range(N_lines):
    ax = axes[0, ii_line]

    for ii_T0, color in enumerate(cmap1):
        ax.plot(Delta_x, transcendental_resolution[ii_line, ii_T0, fixed_n0_idx, :], color=color, linewidth=2.5, linestyle='-')
        ax.axhline(y=-jnp.log(resolution_fraction), color='black', linestyle=':', alpha=0.7, linewidth=1.5)

    ax.set_xlabel(r'$\Delta x$', fontsize=18)
    ax.set_ylabel(r'$f(\Delta x)$', fontsize=18)
    ax.set_xscale('log')
    ax.set_yscale("log")

    # (Optional) label each column with line identifier
    ax.set_title(rf"Line {ii_line+1}", fontsize=18)

# -------------------------
# Bottom row: vary n0 (fix T0)
# -------------------------
for ii_line in range(N_lines):
    ax = axes[1, ii_line]

    for ii_n0, color in enumerate(cmap2):
        ax.plot(Delta_x, transcendental_resolution[ii_line, fixed_T0_idx, ii_n0, :], color=color, linewidth=2.5, linestyle='-')
        ax.axhline(y=-jnp.log(resolution_fraction), color='black', linestyle=':', alpha=0.7, linewidth=1.5)

    ax.set_xlabel(r'$\Delta x$', fontsize=18)
    ax.set_ylabel(r'$f(\Delta x)$', fontsize=18)
    ax.set_xscale('log')
    ax.set_yscale("log")

    ax.set_title(rf"Line {ii_line+1}", fontsize=18)

# -------------------------------------------------------
# Add legends (once) — attach to representative axes
# -------------------------------------------------------
ax_top_ref = axes[0, -1]     # rightmost top axis
ax_bot_ref = axes[1, -1]     # rightmost bottom axis

# Top row legend: T0 color mapping
leg = ax_top_ref.legend(handles=handles_T0, loc="lower right", fontsize=14, ncols=1, frameon=True)
ax_top_ref.add_artist(leg)

# Bottom row legend: n0 color mapping
leg = ax_bot_ref.legend(handles=handles_n0, loc="lower right", fontsize=14, ncols=1, frameon=True)
ax_bot_ref.add_artist(leg)

fig.tight_layout()
plt.show()